In [ ]:
from __future__ import print_function
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from keras import regularizers
from keras.models import Model
from keras.models import Sequential
from keras.layers import *
from keras.regularizers import l1, l2
import keras
import keras.utils as kutils
from keras.optimizers import SGD
from keras.callbacks import EarlyStopping, Callback, ModelCheckpoint,ReduceLROnPlateau
from scipy.stats import pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow.keras.backend as K


In [ ]:
##one of K encoding
nb_classes = 4

def indices_to_one_hot(data,nb_classes):
	
	targets = np.array(data).reshape(-1)
	
	return np.eye(nb_classes)[targets]
	

In [ ]:
def readData(input):
    # Read the data
    data = pd.read_csv(input, sep='\t', header=0, na_values='nan')

    # Convert SNP, pheno, and folds columns to numeric, coercing errors to NaN
    SNP = data.iloc[:, 4:].apply(pd.to_numeric, errors='coerce').values
    pheno = pd.to_numeric(data.iloc[:, 1], errors='coerce').values
    folds = pd.to_numeric(data.iloc[:, 0], errors='coerce').values

    # Initialize the array to store one-hot encoded SNP data
    arr = np.empty(shape=(SNP.shape[0], SNP.shape[1], nb_classes))
    
    # Iterate over the rows to convert SNP values to one-hot encoding
    for i in range(0, SNP.shape[0]):
        arr[i] = indices_to_one_hot(pd.to_numeric(SNP[i], downcast='signed'), nb_classes)
    
    return arr, pheno, folds

In [ ]:
def resnet(input):

	inputs = Input(shape=(input.shape[1],nb_classes))


	x = Conv1D(10,4,padding='same',activation = 'linear',kernel_initializer = 'TruncatedNormal', kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(inputs)

	x = Conv1D(10,20,padding='same',activation = 'linear', kernel_initializer = 'TruncatedNormal',kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(x)

	x = Dropout(0.75)(x)

	shortcut = Conv1D(10,4,padding='same',activation = 'linear',kernel_initializer = 'TruncatedNormal', kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(inputs)
	x = layers.add([shortcut,x])

	x = Conv1D(10,4,padding='same',activation = 'linear',kernel_initializer = 'TruncatedNormal', kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(x)

	x = Dropout(0.75)(x)
	x = Flatten()(x)

	x = Dropout(0.75)(x)

	outputs = Dense(1,activation = isru,bias_regularizer = regularizers.l2(0.01),kernel_initializer = 'TruncatedNormal',name = 'out')(x)

	model = Model(inputs = inputs,outputs = outputs)
	model.compile(loss='mean_squared_error',optimizer=keras.optimizers.Adam(learning_rate=0.001),metrics=['mae'])

	return model


In [ ]:
def compile_saliency_function(model):
	
	inp = model.layers[0].input
	outp = model.layers[10].output
	max_outp = K.max(outp, axis=1)
	saliency = K.gradients(K.sum(max_outp), inp)
	return K.function([inp,K.learning_phase()], saliency)

In [ ]:
def show_images_plot(saliency, wald, outname):
	
	plt.figure(figsize=(15, 8), facecolor='w')
	
	plt.subplot(2, 1, 1)
	x = np.median(saliency,axis=-1)
	plt.plot(x,'b.')
	line = sorted(x,reverse = True)[10]
	plt.axhline(y = line,color='b', linestyle='--')
	plt.ylabel('saliency value', fontdict=None, labelpad=None,fontsize=15)
	
	
	plt.subplot(2, 1, 2)
	plt.plot(wald,'r1')
	line = sorted(wald,reverse = True)[10]
	plt.axhline(y = line,color='r', linestyle='--')
	
	plt.xlabel('SNPs', fontdict=None, labelpad=None,fontsize=15)
	plt.ylabel('Wald', fontdict=None, labelpad=None,fontsize=15)
	
	plt.savefig(outname)
	plt.clf()
	plt.cla()
	plt.close()	

In [ ]:
def get_saliency(testSNP, model):
	
	array= np.array([testSNP])
	saliency_fn = compile_saliency_function(model)
	saliency_out = saliency_fn([[y for y in array][0],1])
	saliency = saliency_out[0]
	saliency = saliency[::-1].transpose(1, 0, 2)
	output= np.abs(saliency).max(axis=-1)
	
	return output

In [ ]:
a= 0.03  #height

def isru(x):
	return  x/(tf.sqrt(1+a*tf.square(x)))

In [47]:
def model_train(test, val, train, testPheno, valPheno, trainPheno, model_save, weights_save):

	batch_size = 250
	early_stop = 5
	epoch = 10
	early_stopping = EarlyStopping(monitor='val_mae', mode='min', patience=early_stop, verbose=1, restore_best_weights=True)

	model = resnet(train)
	history = model.fit(train, trainPheno, batch_size=batch_size, epochs=epoch, validation_data=(val, valPheno), callbacks=[early_stopping], shuffle= True)

	model.save(model_save)
	model.save_weights(weights_save)

	pred = model.predict(test)
	pred.shape = (pred.shape[0],)
	corr = pearsonr(pred,testPheno)[0]

	return history,corr

In [45]:
def main(IMP_input, QA_input):

    IMP_corr = []
    QA_corr = []

    # Read data for imputed (IMP) and non-imputed (QA) datasets
    imp_SNP, imp_pheno, _ = readData(IMP_input)  # We don't need the fold info anymore
    QA_SNP, QA_pheno, _ = readData(QA_input)

    PHENOTYPE = imp_pheno  # Use imputed phenotypes for training

    # Split data into training, validation, and test sets
    # You can adjust the splitting ratios if needed
    train_size = int(0.7 * len(imp_SNP))  # 70% for training
    val_size = int(0.15 * len(imp_SNP))   # 15% for validation
    test_size = len(imp_SNP) - train_size - val_size  # Remaining 15% for testing

    trainSNP, valSNP, testSNP = imp_SNP[:train_size], imp_SNP[train_size:train_size+val_size], imp_SNP[train_size+val_size:]
    trainSNP_QA, valSNP_QA, testSNP_QA = QA_SNP[:train_size], QA_SNP[train_size:train_size+val_size], QA_SNP[train_size+val_size:]
    trainPheno, valPheno, testPheno = PHENOTYPE[:train_size], PHENOTYPE[train_size:train_size+val_size], PHENOTYPE[train_size+val_size:]

    # Train model on imputed dataset
    history, corr = model_train(testSNP, valSNP, trainSNP, testPheno, valPheno, trainPheno, 'model_IMP/model_full.txt', 'model_IMP/model_weights_full.h5')
    IMP_corr.append(float('%0.4f' % corr))

    # Calculate and plot saliency for the imputed dataset
    saliency = get_saliency(testSNP, model)  # Compute saliency
    # Define or calculate the 'wald' values if needed; here it's just a placeholder
    wald = np.random.rand(len(saliency))  # Example placeholder, replace with real calculation
    show_images_plot(saliency, wald, f"saliency_plot_IMP_{i}.png")


    # Train model on non-imputed dataset (QA)
    history, corr = model_train(testSNP_QA, valSNP_QA, trainSNP_QA, testPheno, valPheno, trainPheno, 'model_QA/model_full.txt', 'model_QA/model_weights_full.h5')
    QA_corr.append(float('%0.4f' % corr))

    # Print final results
    print("Average PCC (imputed) from the full dataset: " + str(np.mean(IMP_corr)))
    print("Average PCC (non-imputed) from the full dataset: " + str(np.mean(QA_corr)))


In [ ]:
def main(IMP_input, QA_input):
    IMP_corr=[]
    QA_corr = []

    imp_SNP, imp_pheno, folds = readData(IMP_input)
    QA_SNP, QA_pheno, folds = readData(QA_input)

    PHENOTYPE = imp_pheno

    for i in range(1, 11):
        testIdx = np.where(folds == i)
        if i == 10:
            valIdx = np.where(folds == 1)
            trainIdx = np.intersect1d(np.where(folds != i), np.where(folds != 1))
        else:
            valIdx = np.where(folds == i + 1)
            trainIdx = np.intersect1d(np.where(folds != i), np.where(folds != i + 1))

        trainSNP, trainSNP_QA , trainPheno = imp_SNP[trainIdx], QA_SNP[trainIdx], PHENOTYPE[trainIdx]
        valSNP, valSNP_QA, valPheno = imp_SNP[valIdx], QA_SNP[valIdx], PHENOTYPE[valIdx]
        testSNP, testSNP_QA, testPheno = imp_SNP[testIdx], QA_SNP[testIdx], PHENOTYPE[testIdx]

        # Train model on imputed dataset
        history, corr = model_train(testSNP, valSNP, trainSNP, testPheno, valPheno, trainPheno, 'model_IMP/model_'+str(i)+'.txt', 'model_IMP/model_weights'+str(i)+'.h5')
        IMP_corr.append(float('%0.4f' % corr))

        # Calculate and plot saliency for the imputed dataset
        saliency = get_saliency(testSNP, model)  # Compute saliency
        # Define or calculate the 'wald' values if needed; here it's just a placeholder
        wald = np.random.rand(len(saliency))  # Example placeholder, replace with real calculation
        show_images_plot(saliency, wald, f"saliency_plot_IMP_{i}.png")

        # Train model on non-imputed dataset (QA)
        history, corr = model_train(testSNP_QA, valSNP_QA, trainSNP_QA, testPheno, valPheno, trainPheno, 'model_QA/model_'+str(i)+'.txt', 'model_QA/model_weights'+str(i)+'.h5')
        QA_corr.append(float('%0.4f' % corr))

        # Calculate and plot saliency for the non-imputed dataset
        saliency = get_saliency(testSNP_QA, model)  # Compute saliency
        # Define or calculate the 'wald' values for non-imputed data as well
        wald = np.random.rand(len(saliency))  # Example placeholder, replace with real calculation
        show_images_plot(saliency, wald, f"saliency_plot_QA_{i}.png")

    print("Average PCC (imputed) from 10-fold cross validation: " + str(np.mean(IMP_corr)))
    print("Average PCC (non-imputed) from 10-fold cross validation: " + str(np.mean(QA_corr)))

In [48]:
def main(IMP_input, QA_input):

	IMP_corr=[]
	QA_corr = []

	imp_SNP,imp_pheno, folds = readData(IMP_input)
	QA_SNP,QA_pheno, folds = readData(QA_input)


	PHENOTYPE = imp_pheno

	for i in range(1,11):

		testIdx = np.where(folds == i)
		if i == 10:
			valIdx = np.where(folds == 1)
			trainIdx = np.intersect1d(np.where(folds != i),np.where(folds != 1))
		else:
			valIdx = np.where(folds == i+1)
			trainIdx = np.intersect1d(np.where(folds != i),np.where(folds != i+1))

		trainSNP, trainSNP_QA , trainPheno = imp_SNP[trainIdx], QA_SNP[trainIdx], PHENOTYPE[trainIdx]
		valSNP, valSNP_QA, valPheno = imp_SNP[valIdx],QA_SNP[valIdx], PHENOTYPE[valIdx]
		testSNP, testSNP_QA, testPheno = imp_SNP[testIdx],QA_SNP[testIdx], PHENOTYPE[testIdx]

		history, corr = model_train(testSNP,valSNP,trainSNP,testPheno,valPheno,trainPheno,'model_IMP/model_'+str(i)+'.txt','model_IMP/model_weights'+str(i)+'.h5')
		IMP_corr.append(float('%0.4f' % corr))

		history, corr = model_train(testSNP_QA,valSNP_QA,trainSNP_QA,testPheno,valPheno,trainPheno,'model_QA/model_'+str(i)+'.txt','model_QA/model_weights'+str(i)+'.h5')
		QA_corr.append(float('%0.4f' % corr))

	print ("Average PCC (imputed) from 10-fold cross validation: " + str(np.mean(IMP_corr)))
	print ("Average PCC (non-imputed) from 10-fold cross validation: " + str(np.mean(QA_corr)))

In [49]:
if __name__ == '__main__':
	
	#os.chdir("MOISTURE")
	
	IMP_input =  "IMP_height.txt"
	QA_input = "QA_height.txt"
	
	main(IMP_input,QA_input)


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 787ms/step - loss: 2.3300 - mae: 1.0605 - val_loss: 1.4541 - val_mae: 0.7789
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 14s 817ms/step - loss: 1.5405 - mae: 0.8150 - val_loss: 1.3836 - val_mae: 0.7697
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 1.4270 - mae: 0.7957 - val_loss: 1.3325 - val_mae: 0.7665
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 869ms/step - loss: 1.3799 - mae: 0.7882 - val_loss: 1.2820 - val_mae: 0.7644
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 900ms/step - loss: 1.2593 - mae: 0.7573 - val_loss: 1.2366 - val_mae: 0.7606
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 865ms/step - loss: 1.2908 - mae: 0.7893 - val_loss: 1.1936 - val_mae: 0.7554
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 909ms/step - loss: 1.2291 - mae: 0.7708 - val_loss: 1.1584 - val_mae: 0.7529
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 16s 910ms/step - loss: 1.1659 - mae: 0.7525 - val_loss: 1.1136 - val_mae: 0.7434
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 857ms/ste

the average Pearson Correlation Coefficient (PCC) obtained from a 10-fold cross-validation process. PCC is a measure of the linear correlation between two variables, ranging from -1 (perfect negative correlation) to 1 (perfect positive correlation). A value closer to 0 indicates no linear correlation.
	•	Imputed Data (0.4175): This value suggests a moderate positive correlation between the predicted and actual values when using imputed data.
	•	Non-Imputed Data (0.52877): This value indicates a stronger positive correlation for the non-imputed data.

These results imply that your model performs better when using non-imputed data compared to imputed data.
